## 1. IMPORT NECESSARY LIBRARY

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.formula.api as smf

Read 'final' file

In [ ]:
df =  pd.read_csv('../data/processed/final.csv')

## 2. EXPLORING DATA ANALYSIS

The primary objective of the project is to optimize the %dry results of the experiments, therefore, exploration of this factor will be conducted.

In [ ]:
df[['% dry after 24h in humidity', '% dry after 2 days in humidity chamber']].describe()

In [ ]:
sns.histplot(df, x = '% dry after 24h in humidity', kde=True, bins=30, color="skyblue", edgecolor="black", alpha=0.7)
plt.title("Histogram of %dry 24h", fontsize=14)
plt.show()

In [ ]:
sns.histplot(df, x = '% dry after 2 days in humidity chamber', kde=True, bins=30, color="skyblue", edgecolor="black", alpha=0.7)
plt.title("Histogram of %dry 24h", fontsize=14)
plt.show()

The uneven distribution of results is easily recognizable, separating them into two distinct groups.<br>
<br>
The range from 90 to 100 has the largest number, while the range from 0 to 90 is scattered and not concentrated.<br>
<br>
<br>
This uneven distribution of data suggests that the results come from different formulas or processes that are not related to each other.

In [ ]:
sns.boxplot(x=df['% dry after 24h in humidity'])
plt.show()

In [ ]:
sns.boxplot(x=df['% dry after 2 days in humidity chamber'])
plt.show()

Since lower values ​​indicate better performance, the distribution suggests that most experiments did not meet the target. The median is close to 90, and a large proportion of observations are concentrated near the upper end of the scale, indicating generally poor performance with only a small subset of potentially successful runs.

In [ ]:
dfc =  df.select_dtypes(include='number').corr()
cr = dfc[['% dry after 24h in humidity', '% dry after 2 days in humidity chamber']]
cr

In [ ]:
cr.isna().sum().to_frame()

In [ ]:
cr = cr.dropna()

In [ ]:
plt.figure(figsize=(20, 20))
sns.heatmap(cr.astype(float), annot=True, fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
model = smf.ols(formula='Q(\'UV viscosity (cP)\') ~ Q(\'% dry after 24h in humidity\')', data=df).fit()
sns.regplot(x='UV viscosity (cP)', y='% dry after 24h in humidity', data=df)
plt.show()
model.summary()

In [ ]:
model1= smf.ols(formula='Q(\'Emulsion viscosity (cP)\') ~ Q(\'% dry after 24h in humidity\')', data=df).fit()
sns.regplot(x='Emulsion viscosity (cP)', y='% dry after 24h in humidity', data=df)
plt.show()
model1.summary()

In [ ]:
model2= smf.ols(formula='Q(\'ESBOA\') ~ Q(\'% dry after 24h in humidity\')', data=df).fit()
sns.regplot(x='ESBOA', y='% dry after 24h in humidity', data=df)
plt.show()
model2.summary()

In [ ]:
model3= smf.ols(formula='Q(\'TPGDA\') ~ Q(\'% dry after 24h in humidity\')', data=df).fit()
sns.regplot(x='TPGDA', y='% dry after 24h in humidity', data=df)
plt.show()
model3.summary()

In [ ]:
model4= smf.ols(formula='Q(\'SR833S\') ~ Q(\'% dry after 24h in humidity\')', data=df).fit()
sns.regplot(x='SR833S', y='% dry after 24h in humidity', data=df)
plt.show()
model4.summary()

In [ ]:
model5= smf.ols(formula='Q(\'SR399\') ~ Q(\'% dry after 24h in humidity\')', data=df).fit()
sns.regplot(x='SR399', y='% dry after 24h in humidity', data=df)
plt.show()
model5.summary()

In [ ]:
model6= smf.ols(formula = 'Q("% dry after 24h in humidity") ~ Q("SR399") + Q("UV viscosity (cP)") + Q("Emulsion viscosity (cP)") + Q("ESBOA") + Q("TPGDA") + Q("SR833S")', data=df).fit()
model6.summary()

In [ ]:
model_uv = smf.ols(
    formula="""
    Q('UV viscosity (cP)') ~
    Q('SR833S') +
    Q('ESBOA') +
    Q('TPGDA') +
    Q('SR399') +
    Q('Miramer PU2560') +
    Q('Radactive SF') +
    Q('Continuous Flow Rate (mL/min)') +
    Q('Flow Rate Ratio')
    """,
    data=df
).fit()

print(model_uv.summary())

The model explains UV viscosity very well (R² = 0.949, Adjusted R² = 0.946), indicating that the selected formulation variables are appropriate and strongly correlated with UV viscosity. Specifically, SR833S, ESBOA, TPGDA, Miramer PU2560, and Radactive SF all showed clear statistical significance. Conversely, SR399 and Flow Rate Ratio did not show a significant independent contribution to the presence of the remaining variables, while Continuous Flow Rate was only near-significant. However, the warning about multicollinearity suggests that individual coefficients need to be interpreted cautiously.

In [ ]:
model_uv_reduced = smf.ols(
    formula="""
    Q('UV viscosity (cP)') ~
    Q('SR833S') +
    Q('ESBOA') +
    Q('TPGDA') +
    Q('Miramer PU2560') +
    Q('Radactive SF') +
    Q('Continuous Flow Rate (mL/min)')
    """,
    data=df
).fit()

print(model_uv_reduced.summary())

In [ ]:
model_uv_formulation_only = smf.ols(
    formula="""
    Q('UV viscosity (cP)') ~
    Q('SR833S') +
    Q('ESBOA') +
    Q('TPGDA') +
    Q('Miramer PU2560') +
    Q('Radactive SF')
    """,
    data=df
).fit()

print(model_uv_formulation_only.summary())

In [ ]:
model_emulsion = smf.ols(
    formula="""
    Q('Emulsion viscosity (cP)') ~
    Q('UV viscosity (cP)') +
    Q('Core Viscosity (cP)') +
    Q('Amount of Core Used (g)') +
    Q('Core Loading (%)') +
    Q('Continuous Flow Rate (mL/min)') +
    Q('Flow Rate Ratio')
    """,
    data=df
).fit()

print(model_emulsion.summary())

In [ ]:
model_emulsion_reduced = smf.ols(
    formula="""
    Q('Emulsion viscosity (cP)') ~
    Q('UV viscosity (cP)') +
    Q('Amount of Core Used (g)') +
    Q('Core Loading (%)') +
    Q('Continuous Flow Rate (mL/min)') +
    Q('Flow Rate Ratio')
    """,
    data=df
).fit()

print(model_emulsion_reduced.summary())

The reduced regression model explains emulsion viscosity extremely well (R² = 0.955, Adjusted R² = 0.954). UV viscosity remains a strong positive predictor, indicating that the rheological state of the UV phase carries forward into the emulsion stage. In addition, process variables, particularly Continuous Flow Rate and Flow Rate Ratio, show strong and statistically significant effects, highlighting the importance of operating conditions during emulsion formation. Core Loading also shows a significant positive association, while Amount of Core Used shows a significant negative association. Removing Core Viscosity does not reduce model performance, indicating that it does not provide an independent contribution once the remaining predictors are included.

In [ ]:
model_emulsion_inter = smf.ols(
    formula="""
    Q('Emulsion viscosity (cP)') ~
    Q('UV viscosity (cP)') +
    Q('Amount of Core Used (g)') +
    Q('Core Loading (%)') +
    Q('Continuous Flow Rate (mL/min)') +
    Q('Flow Rate Ratio') +
    Q('UV viscosity (cP)'):Q('Continuous Flow Rate (mL/min)') +
    Q('UV viscosity (cP)'):Q('Flow Rate Ratio')
    """,
    data=df
).fit()

print(model_emulsion_inter.summary())

The interaction model improves the explanation of emulsion viscosity compared with the additive model. In particular, the effect of UV viscosity on emulsion viscosity depends on process conditions. The negative interaction between UV viscosity and Continuous Flow Rate suggests that increasing flow rate weakens the positive influence of UV viscosity on emulsion viscosity. In contrast, the positive interaction between UV viscosity and Flow Rate Ratio indicates that higher flow rate ratio amplifies the effect of UV viscosity on emulsion viscosity. This suggests that emulsion formation is governed not only by the rheological state of the UV phase, but also by how that state interacts with operating conditions.

In [ ]:
model_dry_near = smf.ols(
    formula="""
    Q('% dry after 24h in humidity') ~
    Q('UV viscosity (cP)') +
    Q('Emulsion viscosity (cP)') +
    Q('Amount of Core Used (g)') +
    Q('Core Loading (%)') +
    Q('Continuous Flow Rate (mL/min)') +
    Q('Flow Rate Ratio')
    """,
    data=df
).fit()

print(model_dry_near.summary())

Model 3A shows that % dry after 24h is significantly associated with near-output variables, including UV viscosity, emulsion viscosity, Continuous Flow Rate, and Flow Rate Ratio. UV viscosity shows a negative association, whereas emulsion viscosity and Continuous Flow Rate show positive associations with % dry after 24h, while Flow Rate Ratio shows a negative association. However, the overall explanatory power remains moderate (R² = 0.308), indicating that these variables capture only part of the drying mechanism.

In [ ]:
model_dry_near_1 = smf.ols(
    formula="""
    Q('% dry after 24h in humidity') ~
    Q('UV viscosity (cP)') +
    Q('Emulsion viscosity (cP)') +
    Q('Continuous Flow Rate (mL/min)') +
    Q('Flow Rate Ratio')
    """,
    data=df
).fit()

print(model_dry_near_1.summary())

After removing non-significant variables, the reduced model suggests that % dry after 24h is primarily associated with emulsion viscosity, Continuous Flow Rate, and Flow Rate Ratio. Emulsion viscosity and Continuous Flow Rate show positive associations with % dry after 24h, while Flow Rate Ratio shows a negative association. UV viscosity still shows a negative trend but remains only marginally significant. Although the model is statistically significant overall, its explanatory power remains moderate, indicating that drying performance is influenced by additional factors beyond the current near-output variables.

In [ ]:
model_dry_upstream = smf.ols(
    formula="""
    Q('% dry after 24h in humidity') ~
    Q('SR833S') +
    Q('ESBOA') +
    Q('TPGDA') +
    Q('Miramer PU2560') +
    Q('Radactive SF')
    """,
    data=df
).fit()

print(model_dry_upstream.summary())

In [ ]:
model_dry_upstream_reduced = smf.ols(
    formula="""
    Q('% dry after 24h in humidity') ~
    Q('SR833S') +
    Q('ESBOA') +
    Q('TPGDA')
    """,
    data=df
).fit()

print(model_dry_upstream_reduced.summary())

Model 3B indicates that the upstream UV formulation is meaningfully associated with % dry after 24h, with stronger explanatory power than the near-output model. In particular, SR833S, ESBOA, and TPGDA show significant positive associations with % dry after 24h, while Miramer PU2560 and Radactive SF do not show clear independent contributions. This suggests that the intrinsic UV formulation plays an important role in determining 24-hour drying performance. However, the strong multicollinearity warning means that individual coefficients should be interpreted with caution.

The % dry after 24 hours strongly reflects the initial formulation, rather than just the state near the outlet.

In [ ]:
model_dry_bridge = smf.ols(
    formula="""
    Q('% dry after 24h in humidity') ~
    Q('UV viscosity (cP)') +
    Q('Emulsion viscosity (cP)') +
    Q('SR833S') +
    Q('ESBOA') +
    Q('TPGDA')
    """,
    data=df
).fit()

print(model_dry_bridge.summary())

UV formulation affects the % dry after 24 hours, but that effect is largely transferred through the rheological states of the system, especially UV viscosity and emulsion viscosity.

In [ ]:
model_dry_vis_only = smf.ols(
    formula="""
    Q('% dry after 24h in humidity') ~
    Q('UV viscosity (cP)') +
    Q('Emulsion viscosity (cP)')
    """,
    data=df
).fit()

print(model_dry_vis_only.summary())

In [ ]:
model_dry_bridge_tpgda = smf.ols(
    formula="""
    Q('% dry after 24h in humidity') ~
    Q('UV viscosity (cP)') +
    Q('Emulsion viscosity (cP)') +
    Q('TPGDA')
    """,
    data=df
).fit()

print(model_dry_bridge_tpgda.summary())

The analyses suggest that % dry after 24h is influenced by both upstream UV formulation and downstream emulsion-stage conditions. Among the upstream factors, SR833S, ESBOA, and TPGDA show meaningful associations with % dry. Among the near-output variables, emulsion viscosity, continuous flow rate, and flow rate ratio show the most consistent relationships. The combined model further suggests that the effect of formulation on % dry is likely transmitted largely through viscosity-related system states, especially emulsion viscosity.

In [ ]:
model_dry_1 = smf.ols(
    formula="""
    Q('% dry after 24h in humidity') ~
    Q('Outer viscosity (cP)') +
    Q('Emulsion viscosity (cP)')
    """,
    data=df
).fit()

print(model_dry_1.summary())

In [ ]:
model_dry_2 = smf.ols(
    formula="""
    Q('% dry after 24h in humidity') ~
    Q('Outer viscosity (cP)') +
    Q('Emulsion viscosity (cP)') +
    Q('Continuous Flow Rate (mL/min)') +
    Q('Flow Rate Ratio')
    """,
    data=df
).fit()

print(model_dry_2.summary())

In [ ]:
model_dry_3 = smf.ols(
    formula="""
    Q('% dry after 24h in humidity') ~
    Q('Outer viscosity (cP)') +
    Q('Emulsion viscosity (cP)') +
    Q('Continuous Flow Rate (mL/min)') +
    Q('Flow Rate Ratio') +
    Q('SR833S') +
    Q('TPGDA')
    """,
    data=df
).fit()

print(model_dry_3.summary())

Emulsion viscosity may not be a strong, independent driver as one might imagine, but rather only partially reflects formulation information.

In [ ]:
model_dry_best_reduced = smf.ols(
    formula="""
    Q('% dry after 24h in humidity') ~
    Q('Outer viscosity (cP)') +
    Q('Flow Rate Ratio') +
    Q('SR833S') +
    Q('TPGDA')
    """,
    data=df
).fit()

print(model_dry_best_reduced.summary())

The reduced model suggests that % dry after 24h is best explained by a combination of upstream formulation and near-output system state. In particular, SR833S and TPGDA show strong positive associations with % dry, while outer viscosity shows a significant negative association. Flow rate ratio also remains significant in the reduced model, indicating that process conditions continue to matter alongside formulation effects.